[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/36_int8_quantization.ipynb)

# 🔴 Hard: INT8 Quantized Linear

*Inference & Decoding*
Implement an **INT8-quantized linear layer** using symmetric per-channel
quantization.

$$s_o = \frac{\max_i |W_{oi}|}{127}, \qquad
  W^{q}_{oi} = \mathrm{clip}\big(\mathrm{round}(W_{oi}/s_o),\ -128,\ 127\big)$$

and at inference $\hat{W} = W^{q} \cdot s$, then $y = x\hat{W}^\top + b$.

### Signature
```python
class Int8Linear(nnx.Module):
    def __init__(self, weight, bias=None): ...   # weight: (out, in)
    def __call__(self, x): ...
```

### Rules
- **Symmetric**: zero maps to zero, so there is no zero-point
- **Per output channel**: one scale per row of `weight`, shape `(out, 1)`
- Quantize with `round`, then clip to `[-128, 127]`, then cast to `int8`
- Guard the division with `1e-10` so an all-zero row does not produce `NaN`
- Store `weight_int8` and `scale` as `nnx.Variable` (buffers); `bias` stays an
  `nnx.Param`

### Per-tensor vs per-channel
A single scale for the whole matrix is per-*tensor* quantization. It is cheaper
but fragile: one output channel with an unusually large weight sets the scale
for every channel, and all the small ones collapse into a handful of integer
levels. Per-channel gives each row its own scale, costs one float per row, and
is what makes INT8 weight quantization essentially lossless in practice.

### What actually breaks in LLMs
Weight quantization is the easy half. **Activation** quantization is where INT8
falls over, because transformer activations develop systematic outlier
channels — a few dimensions with magnitudes 100× the rest, appearing past
roughly 6.7B parameters. One outlier sets the scale for the whole tensor and
destroys the rest. That observation is exactly what LLM.int8() addresses, by
keeping outlier channels in fp16 and quantizing only the well-behaved ones.

### Why dequantize at all
This implementation stores int8 and computes in float — the win is **memory**
and bandwidth (4× smaller weights), not arithmetic. True int8 matmul with int32
accumulation needs hardware support and a fused kernel; the dequantize-then-
matmul form is what you write when you are showing you understand the numerics.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp
from flax import nnx


class Int8Linear(nnx.Module):
    """Linear layer holding INT8 weights plus a per-channel scale."""

    def __init__(self, weight, bias=None):
        """Args:
            weight: (out_features, in_features) float array to quantize
            bias:   (out_features,) or None
        """
        pass  # Replace this

    def __call__(self, x):
        """(..., in_features) -> (..., out_features)"""
        pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp
from flax import nnx

w = jax.random.normal(jax.random.key(0), (16, 32))
layer = Int8Linear(w)

x = jax.random.normal(jax.random.key(1), (4, 32))
exact = x @ w.T
approx = layer(x)

print("int8 dtype :", layer.weight_int8[...].dtype)
print("scale shape:", layer.scale[...].shape, "(one per output channel)")
print("rel error  :", float(jnp.abs(approx - exact).max() / jnp.abs(exact).max()))
print("memory     : 4x smaller weights (int8 vs float32)")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("int8_quantization")

# hint("int8_quantization")      # stuck? nudge without the answer
# solution("int8_quantization")  # spoiler: the reference implementation